# DualAQI: Objective 1 visualisations

Objective 1: develop **DualAQI**, a dual-branch LSTM (sequence branch: PM2.5 + ERA5 meteorology;
event branch: Punjab stubble-burning fires + calendar; gated fusion) for 24, 48 and 72 h AQI forecasts in Delhi.

Every figure is saved as a 300-dpi PNG and a vector PDF in `results/figures/objective1/`.
Run from the project folder with the `.venv` kernel. Sections: **1 Datasets**, **2 Model**, **3 Results**.

In [ ]:
import glob, json, os, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import FancyBboxPatch, Rectangle, Polygon as MplPolygon
from IPython.display import display
warnings.filterwarnings('ignore')
import dualaqi as D

OUT = 'results/figures/objective1'
os.makedirs(OUT, exist_ok=True)

# Palette: reference categorical slots (fixed order), neutral inks, CPCB AQI category colours
INK, INK2, MUTED, GRID, SURF = '#0b0b0b', '#52514e', '#8f8d86', '#e4e3df', '#ffffff'
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = ('#2a78d6', '#eb6834', '#1baf7a', '#eda100',
                                                           '#e87ba4', '#008300', '#6250d6', '#e34948')
MC = {'DualAQI v2': BLUE, 'LSTM (single-branch)': ORANGE, 'LSTM': ORANGE, 'DualAQI v1': AQUA,
      'Ensemble': YELLOW, 'Persistence': MUTED}
OTHER = '#b9b7b0'
CATS = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very poor', 'Severe']
CAT_EDGES = [0, 50, 100, 200, 300, 400, 1000]
CAT_COL = ['#2e9e4f', '#9ccc5a', '#f2d94e', '#f0a03c', '#e0453a', '#8c1d18']
BLUES = LinearSegmentedColormap.from_list('blues', ['#f3f7fd', '#9ec5f4', '#3987e5', '#1c5cab', '#0d366b'])
FIRES = LinearSegmentedColormap.from_list('fires', ['#fde7da', '#f6a77c', '#eb6834', '#b9461a', '#6e2408'])
DIVERGE = LinearSegmentedColormap.from_list('div', ['#1c5cab', '#86b6ef', '#f0efec', '#f1a19d', '#b8302f'])

mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'figure.facecolor': SURF, 'axes.facecolor': SURF,
    'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10.5, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.titlepad': 8, 'axes.labelcolor': INK2, 'axes.labelsize': 9,
    'axes.edgecolor': '#b9b7b0', 'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'legend.frameon': False, 'legend.fontsize': 8.5, 'lines.linewidth': 1.6, 'text.color': INK,
})

def save(fig, name):
    fig.savefig(f'{OUT}/{name}.png', bbox_inches='tight', facecolor=SURF)
    fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight', facecolor=SURF)
    plt.show()

def suptitle(fig, title, sub=None, y=1.0):
    fig.suptitle(title, x=0.01, y=y, ha='left', fontsize=12, fontweight='bold')
    if sub:
        fig.text(0.01, y - 0.035, sub, ha='left', va='top', fontsize=9, color=INK2)

def aqi_bands(ax, top=None, labels=True, alpha=0.10):
    top = top or ax.get_ylim()[1]
    for i in range(6):
        lo, hi = CAT_EDGES[i], min(CAT_EDGES[i + 1], top)
        if lo >= top:
            break
        ax.axhspan(lo, hi, color=CAT_COL[i], alpha=alpha, lw=0, zorder=0)
        if labels and hi - lo > 0.04 * top:
            ax.text(1.005, (lo + hi) / 2, CATS[i], transform=ax.get_yaxis_transform(), va='center',
                    fontsize=7.5, color=INK2)

def table_figure(df, name, title, col_w, row_h=0.42, fs=8.3):
    fig, ax = plt.subplots(figsize=(sum(col_w), row_h * (len(df) + 1) + 0.35))
    ax.axis('off')
    tb = ax.table(cellText=df.values, colLabels=list(df.columns), cellLoc='left', colLoc='left',
                  colWidths=[w / sum(col_w) for w in col_w], bbox=[0, 0, 1, 1])
    tb.auto_set_font_size(False)
    tb.set_fontsize(fs)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor(GRID)
        cell.set_linewidth(0.6)
        cell.PAD = 0.03
        if r == 0:
            cell.set_facecolor('#f0efec')
            cell.set_text_props(weight='bold', color=INK)
        else:
            cell.set_facecolor(SURF if r % 2 else '#fafaf8')
            cell.set_text_props(color=INK)
    ax.set_title(title, loc='left', fontsize=11)
    save(fig, name)

print('figures will be written to', OUT)

## 1. Datasets

In [ ]:
pm = pd.read_csv('data/pm25_hourly.csv', parse_dates=['datetime'], index_col='datetime').loc['2019':'2025']
met = pd.read_csv('data/era5_hourly.csv', parse_dates=['datetime'], index_col='datetime').loc['2019':'2025']
fire = pd.read_csv('data/punjab_fire_daily.csv', parse_dates=['date'], index_col='date').loc['2019':'2025']
pm['aqi'] = D.pm25_to_aqi(pm['pm25'].values)
IST = pd.Timedelta(hours=5, minutes=30)
# hourly labels are UTC availability times; CPCB window start in IST = label + 4 h
ist_start = pm.index + pd.Timedelta(hours=4)
daily_pm = pm['pm25'].groupby(ist_start.normalize()).mean()
daily_aqi = pd.Series(D.pm25_to_aqi(daily_pm.values), index=daily_pm.index).loc['2019':'2025']
print(f'PM2.5: {pm.pm25.notna().sum():,} hours; ERA5: {len(met):,} hours; fires: {len(fire):,} days')

### 1.1 Dataset summary

In [ ]:
summary = pd.DataFrame([
    ['Delhi PM2.5', 'CPCB / DPCC / IITM\nmonitoring network', '36 stations, hourly;\ncity mean of stations',
     '2019-01 to 2025-12', f'{pm.pm25.notna().mean()*100:.2f}% of hours', 'Target (AQI) +\nsequence branch'],
    ['ERA5 meteorology', 'ECMWF ERA5 via Open-Meteo\n(28.75°N, 77.25°E)', 't2m, RH, u10, v10,\nsurface pressure, BLH; hourly',
     '2019-01 to 2025-12', f'{met.notna().all(axis=1).mean()*100:.1f}% of hours\n(BLH gap Jan-Jun 2024)', 'Sequence branch'],
    ['Punjab fires', 'NASA FIRMS VIIRS S-NPP\n375 m active fires', 'Daily count + FRP sum,\nPunjab polygon, vegetation fires',
     '2019-01 to 2025-12', '100% of days', 'Event branch'],
    ['Calendar', 'Derived', 'Hour, weekday, month,\nfestival flags', '2019-01 to 2025-12', 'complete', 'Event branch'],
], columns=['Dataset', 'Source', 'Content', 'Period', 'Completeness', 'Used in'])
display(summary)
table_figure(summary, 'fig01_dataset_summary', 'Table 1. Datasets used for DualAQI',
             col_w=[1.5, 2.1, 2.3, 1.5, 1.7, 1.5], row_h=0.55)

### 1.2 Study area

In [ ]:
from build_inputs import punjab_path
g = json.load(open('raw/boundaries/gadm41_IND_1.json', encoding='utf-8'))
def rings(f):
    geom = f['geometry']
    polys = geom['coordinates'] if geom['type'] == 'MultiPolygon' else [geom['coordinates']]
    return [np.asarray(p[0]) for p in polys]
ext = (73.3, 78.5, 27.7, 33.0)
# fire detections of the 2021 paddy season (Oct-Nov), inside Punjab, vegetation fires
files = sorted(glob.glob('raw/firms/VIIRS_SNPP_SP_2021-09-*.csv') + glob.glob('raw/firms/VIIRS_SNPP_SP_2021-1[01]-*.csv'))
det = pd.concat([pd.read_csv(f) for f in files])
det['acq_date'] = pd.to_datetime(det.acq_date)
det = det[(det.acq_date >= '2021-10-01') & (det.acq_date <= '2021-11-30') & (det['type'] == 0)]
det = det[punjab_path('raw/boundaries/gadm41_IND_1.json').contains_points(det[['longitude', 'latitude']].to_numpy())]

fig, ax = plt.subplots(figsize=(7.4, 7.2))
ax.grid(False)
for f in g['features']:
    name = f['properties']['NAME_1']
    for r in rings(f):
        if r[:, 0].max() < ext[0] or r[:, 0].min() > ext[1] or r[:, 1].max() < ext[2] or r[:, 1].min() > ext[3]:
            continue
        fc = '#dcebfb' if 'Delhi' in name else ('#fbf1ea' if name == 'Punjab' else '#f4f3ef')
        ax.add_patch(MplPolygon(r, closed=True, facecolor=fc, edgecolor='#a9a7a0', lw=0.6, zorder=1))
        if name == 'Punjab':
            ax.add_patch(MplPolygon(r, closed=True, fill=False, edgecolor=INK, lw=1.3, zorder=4))
hb = ax.hexbin(det.longitude, det.latitude, gridsize=55, extent=(73.8, 77.0, 29.5, 32.6), mincnt=1,
               bins='log', cmap=FIRES, linewidths=0, zorder=3)
cb = fig.colorbar(hb, ax=ax, shrink=0.55, pad=0.02)
cb.set_label('Fire detections per cell, Oct-Nov 2021 (log)', color=INK2)
cb.outline.set_visible(False)
ax.add_patch(Rectangle((77.125, 28.625), 0.25, 0.25, fill=False, edgecolor=BLUE, lw=1.3, ls='--', zorder=5))
ax.plot(77.2167, 28.6315, marker='*', ms=15, color=BLUE, mec='white', mew=0.8, zorder=6)
ax.annotate('Delhi (Connaught Place)\nPM2.5: 36 CPCB stations\nERA5 grid cell (dashed)', (77.2167, 28.6315),
            xytext=(77.45, 28.05), fontsize=8, color=INK, arrowprops=dict(arrowstyle='-', color=INK2, lw=0.7), zorder=7)
ax.annotate('', xy=(77.05, 28.88), xytext=(76.05, 29.95),
            arrowprops=dict(arrowstyle='-|>', color=INK2, lw=1.8, mutation_scale=15), zorder=6)
ax.text(76.62, 29.62, 'north-westerly\nsmoke transport', fontsize=8, color=INK, ha='left', va='center', zorder=7,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=GRID, lw=0.6))
for x, y, t, c, w in [(75.2, 31.25, 'PUNJAB', INK, 'bold'), (75.85, 28.95, 'HARYANA', MUTED, 'normal'),
                      (77.55, 31.75, 'HIMACHAL\nPRADESH', MUTED, 'normal'), (74.3, 28.3, 'RAJASTHAN', MUTED, 'normal'),
                      (78.1, 29.35, 'UTTAR\nPRADESH', MUTED, 'normal'), (73.75, 31.75, 'PAKISTAN', MUTED, 'normal')]:
    ax.text(x, y, t, fontsize=8.5, color=c, ha='center', fontweight=w, zorder=6)
lat0 = 28.0
dlon = 100 / (111.32 * np.cos(np.deg2rad(lat0)))
ax.plot([73.55, 73.55 + dlon], [lat0, lat0], color=INK, lw=2.2, zorder=6)
ax.text(73.55 + dlon / 2, lat0 + 0.08, '100 km', ha='center', fontsize=8, zorder=6)
ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
ax.set_aspect(1 / np.cos(np.deg2rad(30.3)))
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.set_title(f'Study area: Delhi and the Punjab stubble-burning region\n'
             f'{len(det):,} VIIRS fire detections in Punjab, Oct-Nov 2021', fontsize=10.5)
save(fig, 'fig02_study_area_map')

In [ ]:
# All Oct-Nov detections 2019-2025 inside Punjab (vegetation fires), reused by the next two maps
PJ = punjab_path('raw/boundaries/gadm41_IND_1.json')
parts = []
for y in range(2019, 2026):
    fs = [f for f in glob.glob(f'raw/firms/VIIRS_SNPP_SP_{y}-*.csv')
          if f'{y}-09-2' in f or f'{y}-10-' in f or f'{y}-11-' in f]
    d = pd.concat([pd.read_csv(f) for f in fs])
    d['acq_date'] = pd.to_datetime(d.acq_date)
    d = d[(d.acq_date >= f'{y}-10-01') & (d.acq_date <= f'{y}-11-30') & (d['type'] == 0)]
    parts.append(d[PJ.contains_points(d[['longitude', 'latitude']].to_numpy())].assign(year=y))
ON = pd.concat(parts, ignore_index=True)
print(ON.groupby('year').size().to_dict())

def base_map(ax, ext, label_states=True, lw=0.5):
    ax.grid(False)
    for f in g['features']:
        name = f['properties']['NAME_1']
        for r in rings(f):
            if r[:, 0].max() < ext[0] or r[:, 0].min() > ext[1] or r[:, 1].max() < ext[2] or r[:, 1].min() > ext[3]:
                continue
            fc = '#dcebfb' if 'Delhi' in name else ('#fbf1ea' if name == 'Punjab' else '#f4f3ef')
            ax.add_patch(MplPolygon(r, closed=True, facecolor=fc, edgecolor='#a9a7a0', lw=lw, zorder=1))
            if name == 'Punjab':
                ax.add_patch(MplPolygon(r, closed=True, fill=False, edgecolor=INK, lw=1.0, zorder=4))
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
    ax.set_aspect(1 / np.cos(np.deg2rad((ext[2] + ext[3]) / 2)))

In [ ]:
from matplotlib.colors import LogNorm
ext_p = (73.8, 77.05, 29.45, 32.6)
norm = LogNorm(1, 150)
fig, axes = plt.subplots(2, 4, figsize=(12.5, 7.4), gridspec_kw=dict(wspace=0.06, hspace=0.18))
for ax, y in zip(axes.ravel(), range(2019, 2026)):
    base_map(ax, ext_p)
    d = ON[ON.year == y]
    hb = ax.hexbin(d.longitude, d.latitude, gridsize=42, extent=ext_p, mincnt=1, norm=norm, cmap=FIRES, linewidths=0, zorder=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f'{y}', fontsize=11, loc='left', pad=2)
    ax.text(0.98, 0.03, f'{len(d)/1000:.1f}k fires', transform=ax.transAxes, ha='right', fontsize=9,
            fontweight='bold', color=INK, bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=GRID, lw=0.6))
last_ax = axes.ravel()[-1]
last_ax.axis('off')
cax = last_ax.inset_axes([0.12, 0.55, 0.76, 0.07])
cb = fig.colorbar(hb, cax=cax, orientation='horizontal')
cb.set_label('fire detections per cell (log)', color=INK2, fontsize=8.5); cb.outline.set_visible(False)
tot = ON.groupby('year').size() / 1000
last_ax.text(0.5, 0.36, f'Oct-Nov detections fell from\n{tot.max():.1f}k ({tot.idxmax()}) to {tot.loc[2025]:.1f}k (2025),\n'
             f'a {100 * (1 - tot.loc[2025] / tot.max()):.0f}% decline', transform=last_ax.transAxes, ha='center',
             va='center', fontsize=9, color=INK)
suptitle(fig, 'Paddy-residue burning in Punjab, October-November of each year',
         'NASA FIRMS VIIRS S-NPP 375 m, vegetation fires inside the Punjab boundary; same colour scale in every panel', y=0.99)
save(fig, 'fig02b_fire_density_by_year')

In [ ]:
def hav_km(lon1, lat1, lon2, lat2):
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.deg2rad(lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))
def dest(lon, lat, brg_deg, km):
    b, d, p = np.deg2rad(brg_deg), km / 6371, np.deg2rad(lat)
    p2 = np.arcsin(np.sin(p) * np.cos(d) + np.cos(p) * np.sin(d) * np.cos(b))
    l2 = np.deg2rad(lon) + np.arctan2(np.sin(b) * np.sin(d) * np.cos(p), np.cos(d) - np.sin(p) * np.sin(p2))
    return np.rad2deg(l2), np.rad2deg(p2)
DL, DT = 77.2167, 28.6315
ON['dist'] = hav_km(DL, DT, ON.longitude, ON.latitude)
ext_s = (72.9, 78.6, 27.6, 33.1)
fig, ax = plt.subplots(figsize=(8.2, 7.6))
base_map(ax, ext_s, lw=0.6)
hb = ax.hexbin(ON.longitude, ON.latitude, C=ON.frp, reduce_C_function=np.sum, gridsize=60, extent=(73.8, 77.05, 29.45, 32.6),
               mincnt=1, bins='log', cmap=FIRES, linewidths=0, zorder=3)
cb = fig.colorbar(hb, ax=ax, shrink=0.5, pad=0.02)
cb.set_label('summed fire radiative power per cell (MW, log)', color=INK2); cb.outline.set_visible(False)
# NW sector seen from Delhi (wind FROM 281-326 deg carries air from Punjab to Delhi)
arc = [dest(DL, DT, b, 480) for b in np.linspace(281.25, 326.25, 40)]
wedge = np.vstack([[DL, DT], np.array(arc), [DL, DT]])
ax.add_patch(MplPolygon(wedge, closed=True, facecolor=BLUE, alpha=0.07, edgecolor=BLUE, lw=0.9, ls='--', zorder=2))
for r in (100, 200, 300, 400):
    ring = np.array([dest(DL, DT, b, r) for b in np.linspace(0, 360, 241)])
    ax.plot(ring[:, 0], ring[:, 1], color=INK2, lw=0.7, ls=(0, (3, 3)), zorder=5)
    lx, ly = dest(DL, DT, 200, r)
    if ext_s[2] < ly < ext_s[3]:
        ax.text(lx, ly, f'{r} km', fontsize=7.8, color=INK2, ha='center', va='center', zorder=6,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.9))
ax.plot(DL, DT, marker='*', ms=15, color=BLUE, mec='white', mew=0.8, zorder=7)
ax.text(DL + 0.12, DT - 0.16, 'Delhi', fontsize=9, fontweight='bold', color=BLUE, zorder=7)
wx, wy = dest(DL, DT, 303.75, 455)
ax.text(wx, wy + 0.08, 'WNW-NW wind sector\n(air from Punjab to Delhi)', fontsize=8, color=BLUE, ha='center', zorder=7,
        bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=GRID, lw=0.6))
med = ON.dist.median(); share300 = (ON.dist <= 300).mean() * 100
ax.text(0.015, 0.015, f'{len(ON):,} fire detections, Oct-Nov 2019-2025\nmedian distance to Delhi: {med:.0f} km\n'
        f'{share300:.0f}% of detections within 300 km', transform=ax.transAxes, fontsize=8.3, va='bottom',
        bbox=dict(boxstyle='round,pad=0.35', fc='white', ec=GRID, lw=0.6), zorder=8)
for x, y, t in [(75.15, 31.3, 'PUNJAB'), (76.0, 28.95, 'HARYANA'), (73.45, 31.9, 'PAKISTAN'), (74.25, 28.15, 'RAJASTHAN')]:
    ax.text(x, y, t, fontsize=8.5, color=INK if t == 'PUNJAB' else MUTED, ha='center',
            fontweight='bold' if t == 'PUNJAB' else 'normal', zorder=6)
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
# inset: Oct-Nov wind directions at Delhi
wdir = (270 - np.degrees(np.arctan2(met.v10, met.u10))) % 360
wd_on = wdir[met.index.month.isin([10, 11])].groupby(met.index[met.index.month.isin([10, 11])].normalize()).agg(
    lambda s: (np.degrees(np.arctan2(np.sin(np.deg2rad(s)).mean(), np.cos(np.deg2rad(s)).mean()))) % 360)
ins = fig.add_axes([0.6, 0.6, 0.17, 0.17], projection='polar')
bins = np.arange(0, 361, 22.5)
cnt, _ = np.histogram((wd_on + 11.25) % 360, bins=bins)
ins.bar(np.deg2rad(bins[:-1]), cnt / cnt.sum() * 100, width=np.deg2rad(22.5) * 0.9,
        color=[ORANGE if 281.25 <= b <= 326.25 else '#9ec5f4' for b in bins[:-1]], edgecolor='white', lw=0.5)
ins.set_theta_zero_location('N'); ins.set_theta_direction(-1)
ins.set_xticks(np.deg2rad([0, 90, 180, 270])); ins.set_xticklabels(['N', 'E', 'S', 'W'], fontsize=7)
ins.set_yticklabels([]); ins.grid(color=GRID, lw=0.5)
ins.set_title('Delhi wind, Oct-Nov\n(direction from)', fontsize=7.5, pad=4, loc='center', fontweight='normal')
ax.set_title('Source and receptor: burning intensity in Punjab and its distance and direction from Delhi', fontsize=10.5)
save(fig, 'fig02c_source_receptor_map')

### 1.3 Data availability

In [ ]:
months = pd.period_range('2019-01', '2025-12', freq='M')
share = lambda s: s.groupby(s.index.to_period('M')).mean().reindex(months)
M = pd.DataFrame({
    'City PM2.5 (hourly)': share(pm.pm25.notna()),
    'ERA5 t2m, RH, wind, pressure': share(met[['t2m', 'rh', 'u10', 'v10', 'sp']].notna().all(axis=1)),
    'ERA5 boundary-layer height': share(met.blh.notna()),
    'Punjab fire counts (daily)': share(fire.fire_count.notna()),
}).T
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 4.6), gridspec_kw=dict(height_ratios=[1.25, 1], hspace=0.55))
im = a1.imshow(M.values * 100, aspect='auto', cmap=BLUES, vmin=0, vmax=100, interpolation='nearest')
a1.set_yticks(range(len(M))); a1.set_yticklabels(M.index)
yr = [i for i, p in enumerate(months) if p.month == 1]
a1.set_xticks(yr); a1.set_xticklabels([months[i].year for i in yr])
a1.grid(False)
for s in a1.spines.values():
    s.set_visible(False)
for i in range(len(months) + 1):
    a1.axvline(i - 0.5, color='white', lw=0.6)
for j in range(len(M) + 1):
    a1.axhline(j - 0.5, color='white', lw=2)
cb = fig.colorbar(im, ax=a1, pad=0.01, aspect=12)
cb.set_label('% present', color=INK2); cb.outline.set_visible(False)
a1.set_title('a  Monthly completeness of each input')
a1.text(1.0, 1.04, 'white cells: boundary-layer height not available in the source, Jan-Jun 2024',
        transform=a1.transAxes, ha='right', va='bottom', fontsize=8, color=INK2)
ns = pm.n_stations.groupby(pm.index.to_period('M')).mean().reindex(months)
a2.plot(months.to_timestamp(), ns.values, color=BLUE)
a2.set_ylim(0, 40); a2.set_ylabel('stations')
a2.set_title('b  Mean number of CPCB stations reporting PM2.5 each hour')
a2.xaxis.set_major_locator(mdates.YearLocator()); a2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
save(fig, 'fig03_data_availability')

### 1.4 PM2.5 / AQI in Delhi

In [ ]:
fig = plt.figure(figsize=(11, 7.2))
gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], hspace=0.42, wspace=0.22)
a = fig.add_subplot(gs[0, :])
a.plot(daily_aqi.index, daily_aqi.values, color='#9a988f', lw=0.6, label='daily mean')
a.plot(daily_aqi.index, daily_aqi.rolling(30, center=True, min_periods=20).mean(), color=BLUE, lw=1.8,
       label='30-day running mean')
a.set_ylim(0, 600); aqi_bands(a, 600)
a.set_ylabel('AQI'); a.legend(loc='upper left', ncol=2)
a.xaxis.set_major_locator(mdates.YearLocator()); a.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
a.set_title('a  Daily city-mean AQI, 2019-2025')
b = fig.add_subplot(gs[1, 0])
data = [daily_aqi[daily_aqi.index.month == m].values for m in range(1, 13)]
bp = b.boxplot(data, whis=(5, 95), showfliers=False, patch_artist=True, widths=0.6,
               medianprops=dict(color=INK, lw=1.4), boxprops=dict(facecolor='#cde2fb', edgecolor=BLUE, lw=0.8),
               whiskerprops=dict(color=BLUE, lw=0.8), capprops=dict(color=BLUE, lw=0.8))
b.set_xticks(range(1, 13)); b.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
b.set_ylim(0, 600); aqi_bands(b, 600, labels=False)
b.set_ylabel('daily AQI'); b.set_title('b  Seasonal cycle (box: IQR, whiskers: 5-95%)')
c = fig.add_subplot(gs[1, 1])
hour = (pm.index + pd.Timedelta(hours=4)).hour
seasons = {'Winter (Dec-Feb)': [12, 1, 2], 'Pre-monsoon (Mar-May)': [3, 4, 5],
           'Monsoon (Jun-Sep)': [6, 7, 8, 9], 'Post-monsoon (Oct-Nov)': [10, 11]}
mon = (pm.index + pd.Timedelta(hours=4)).month
for (name, ms), col in zip(seasons.items(), [BLUE, ORANGE, AQUA, YELLOW]):
    prof = pm.aqi[np.isin(mon, ms)].groupby(hour[np.isin(mon, ms)]).mean()
    c.plot(prof.index, prof.values, color=col, label=name, marker='o', ms=2.5)
c.set_xticks(range(0, 24, 3)); c.set_xlabel('hour of day (IST)'); c.set_ylabel('mean AQI')
c.set_title('c  Diurnal cycle by season'); c.legend(loc='upper center', ncol=2, fontsize=7.8)
c.set_ylim(0, None)
suptitle(fig, 'Delhi air quality, 2019-2025', 'City mean of 36 CPCB stations; AQI from hourly PM2.5 (dualaqi.py mapping)', y=0.99)
save(fig, 'fig04_pm25_overview')

In [ ]:
cat = pd.Series(np.digitize(daily_aqi.values, CAT_EDGES[1:-1]), index=daily_aqi.index)
tab = cat.groupby(cat.index.year).value_counts(normalize=True).unstack(fill_value=0).reindex(columns=range(6), fill_value=0) * 100
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.grid(False)
left = np.zeros(len(tab))
for k in range(6):
    ax.barh(tab.index.astype(str), tab[k], left=left, color=CAT_COL[k], edgecolor='white', lw=1.2, label=CATS[k], height=0.7)
    for y, (l, w) in enumerate(zip(left, tab[k])):
        if w >= 4:
            ax.text(l + w / 2, y, f'{w:.0f}%', ha='center', va='center', fontsize=7.6,
                    color='white' if k in (0, 4, 5) else INK)
    left += tab[k].values
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel('% of days')
for s in ('left', 'bottom'):
    ax.spines[s].set_visible(False)
ax.tick_params(left=False)
ax.legend(ncol=6, loc='upper center', bbox_to_anchor=(0.5, -0.2), handlelength=1.2, columnspacing=1.4)
ax.set_title('Share of days in each AQI category, by year (daily city-mean AQI)')
save(fig, 'fig05_aqi_categories_by_year')

### 1.5 Punjab stubble-burning fires

In [ ]:
fig = plt.figure(figsize=(11, 7.0))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.45, wspace=0.25)
a = fig.add_subplot(gs[0, :])
a.fill_between(fire.index, fire.fire_count, color=ORANGE, lw=0, alpha=0.85)
pk = fire.fire_count.idxmax()
a.set_ylim(0, fire.fire_count.max() * 1.15)
a.annotate(f'peak: {fire.fire_count.max():,} detections ({pk:%d %b %Y})', (pk, fire.fire_count.max()),
           xytext=(pk + pd.Timedelta(days=60), fire.fire_count.max() * 1.05), fontsize=8, va='center',
           arrowprops=dict(arrowstyle='-', color=INK2, lw=0.7, shrinkB=2))
a.set_xlim(fire.index.min() - pd.Timedelta(days=20), fire.index.max() + pd.Timedelta(days=20))
a.set_ylabel('detections per day')
a.xaxis.set_major_locator(mdates.YearLocator()); a.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
a.set_title('a  Daily VIIRS fire detections in Punjab')
b = fig.add_subplot(gs[1, 0])
yrs = fire.index.year
on = fire.fire_count[fire.index.month.isin([10, 11])].groupby(yrs[fire.index.month.isin([10, 11])]).sum() / 1000
rest = fire.fire_count.groupby(yrs).sum() / 1000 - on
b.bar(on.index, on.values, color=ORANGE, width=0.65, label='Oct-Nov (paddy residue)')
b.bar(rest.index, rest.values, bottom=on.values, color='#d9d7d0', width=0.65, edgecolor='white', lw=1.2, label='rest of year')
for x, v, t in zip(on.index, on.values, (on + rest).values):
    b.text(x, t + 1.5, f'{t:.0f}k', ha='center', fontsize=8)
b.set_ylabel('detections (thousands)'); b.set_xticks(on.index); b.legend(loc='upper right')
b.set_title('b  Annual totals'); b.set_ylim(0, (on + rest).max() * 1.15)
c = fig.add_subplot(gs[1, 1])
doy = fire.index.dayofyear
clim = fire.fire_count.groupby(doy).median()
lo = fire.fire_count.groupby(doy).quantile(0.1); hi = fire.fire_count.groupby(doy).quantile(0.9)
c.fill_between(clim.index, lo, hi, color=ORANGE, alpha=0.22, lw=0, label='10-90% across years')
c.plot(clim.index, clim.values, color=ORANGE, label='median')
top = hi.max() * 1.22
for s0, s1, lab in [(91, 151, 'wheat residue\n(Apr-May)'), (274, 334, 'paddy residue\n(Oct-Nov)')]:
    c.axvspan(s0, s1, color='#f0efec', zorder=0, lw=0)
    c.text((s0 + s1) / 2, top * 0.985, lab, ha='center', va='top', fontsize=7.8, color=INK2)
c.set_ylim(0, top)
c.set_xticks([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335])
c.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
c.set_ylabel('detections per day'); c.legend(loc='center', bbox_to_anchor=(0.6, 0.55))
c.set_title('c  Seasonal pattern, 2019-2025')
suptitle(fig, 'Punjab stubble-burning activity (NASA FIRMS VIIRS S-NPP 375 m)',
         'Detections inside the Punjab state boundary, vegetation fires (type 0), all confidence levels', y=0.99)
save(fig, 'fig06_punjab_fires')

In [ ]:
on_mask = daily_aqi.index.month.isin([10, 11])
fire7 = fire.fire_count.rolling(7).sum().shift(1)           # previous 7 days, available by day D
pair = pd.DataFrame({'aqi': daily_aqi, 'fire7': fire7.reindex(daily_aqi.index)}).dropna()
pair = pair[pair.index.month.isin([10, 11])]
per_year = pair.groupby(pair.index.year).apply(lambda d: d.aqi.corr(d.fire7, method='spearman'))
season = pd.DataFrame({'fires_k': on, 'aqi': daily_aqi[on_mask].groupby(daily_aqi.index[on_mask].year).mean()})
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw=dict(wspace=0.28))
a.grid(True, axis='both')
a.plot(season.fires_k, season.aqi, color=GRID, lw=1.2, zorder=1)
a.scatter(season.fires_k, season.aqi, s=55, color=BLUE, edgecolor='white', lw=1.2, zorder=3)
for y, r in season.iterrows():
    a.annotate(str(y), (r.fires_k, r.aqi), xytext=(6, 4), textcoords='offset points', fontsize=8.5)
a.set_xlabel('Oct-Nov fire detections in Punjab (thousands)'); a.set_ylabel('mean daily AQI in Delhi, Oct-Nov')
a.set_title('a  Burning season: fires vs Delhi AQI, by year')
b.bar(per_year.index, per_year.values, color=BLUE, width=0.65)
for x, v in per_year.items():
    b.text(x, v + 0.02 if v >= 0 else v - 0.06, f'{v:.2f}', ha='center', fontsize=8)
b.axhline(0, color=INK2, lw=0.8)
b.set_ylim(min(-0.1, per_year.min() - 0.1), 1); b.set_xticks(per_year.index)
b.set_ylabel('Spearman correlation')
b.set_title('b  Daily AQI vs previous-7-day fire total (Oct-Nov)')
save(fig, 'fig07_fire_aqi_link')

### 1.6 Meteorology (ERA5)

In [ ]:
dm = met.groupby(met.index.normalize()).mean()
dm['T'] = dm.t2m - 273.15
dm['ws'] = np.hypot(met.u10, met.v10).groupby(met.index.normalize()).mean()
dm['tr'] = D.daily_transport(met).reindex(dm.index)
panels = [('T', 'Air temperature (°C)'), ('rh', 'Relative humidity (%)'), ('ws', 'Wind speed at 10 m (m/s)'),
          ('tr', 'NW-to-SE wind component (m/s)'), ('sp', 'Surface pressure (hPa)'), ('blh', 'Boundary-layer height (m)')]
fig, axes = plt.subplots(2, 3, figsize=(11, 5.6), gridspec_kw=dict(hspace=0.5, wspace=0.28))
for ax, (col, title) in zip(axes.ravel(), panels):
    s = dm[col].dropna(); m = s.index.month
    med, q1, q3 = s.groupby(m).median(), s.groupby(m).quantile(0.25), s.groupby(m).quantile(0.75)
    ax.fill_between(med.index, q1, q3, color=BLUE, alpha=0.18, lw=0)
    ax.plot(med.index, med.values, color=BLUE, marker='o', ms=3)
    ax.axvspan(9.5, 11.5, color='#f0efec', zorder=0, lw=0)
    if col == 'tr':
        ax.axhline(0, color=INK2, lw=0.8)
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    ax.set_title(title, fontsize=9.5)
suptitle(fig, 'ERA5 meteorology at Delhi: monthly median of daily means (band: IQR)',
         'Grey band: Oct-Nov burning season. Positive NW-to-SE component = air moving from Punjab towards Delhi.', y=1.02)
save(fig, 'fig08_meteorology_climatology')

In [ ]:
wd = (270 - np.degrees(np.arctan2(dm.v10, dm.u10))) % 360   # direction the wind blows FROM
bins = np.arange(0, 361, 22.5)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.4), subplot_kw=dict(projection='polar'))
for ax, (label, months_) in zip(axes, [('Oct-Nov (burning season)', [10, 11]), ('Jun-Sep (monsoon)', [6, 7, 8, 9])]):
    w = wd[dm.index.month.isin(months_)].dropna()
    cnt, _ = np.histogram((w + 11.25) % 360, bins=bins)
    share_ = cnt / cnt.sum() * 100
    theta = np.deg2rad(bins[:-1])
    cols = [ORANGE if 281.25 <= b <= 326.25 else BLUE for b in bins[:-1]]
    ax.bar(theta, share_, width=np.deg2rad(22.5) * 0.92, color=cols, edgecolor='white', lw=0.8)
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_yticklabels([]); ax.grid(color=GRID)
    ax.set_title(f'{label}\n{share_[[12, 13, 14]].sum():.0f}% of days from WNW-NW', fontsize=9.5, pad=14)
fig.suptitle('Daily mean wind direction at Delhi (ERA5, 2019-2025); orange: sectors facing Punjab',
             x=0.02, ha='left', fontsize=11, fontweight='bold', y=1.04)
save(fig, 'fig09_wind_direction')

### 1.7 Model input features

In [ ]:
feat = pd.DataFrame([
    ['Sequence', 'AQI now', 'City PM2.5 -> AQI (forward-filled <= 3 h)', '168 hourly steps', 'standardised'],
    ['Sequence', 'Wind speed', 'sqrt(u10² + v10²), ERA5', '168 hourly steps', 'standardised'],
    ['Sequence', 't2m, RH', 'ERA5 2 m temperature (K); RH from dewpoint', '168 hourly steps', 'standardised'],
    ['Sequence', 'u10, v10', 'ERA5 10 m wind components (m/s)', '168 hourly steps', 'standardised'],
    ['Sequence', 'Surface pressure', 'ERA5 (hPa)', '168 hourly steps', 'standardised'],
    ['Sequence', 'Boundary-layer height', 'ERA5 BLH (m)', '168 hourly steps', 'standardised'],
    ['Event', 'Fire count', 'Punjab VIIRS detections, days D-1 ... D-7', '7 values', 'log(1+x), standardised'],
    ['Event (v2)', 'Fire radiative power', 'Daily FRP sum (MW), days D-1 ... D-7', '7 values', 'log(1+x), standardised'],
    ['Event', 'Hour, weekday, month', 'Of the forecast origin', '6 values', 'sin / cos'],
    ['Event', 'Festival flags', 'Origin day and the 3 target days', '4 values', '0 / 1'],
    ['Target', 'AQI at +24, +48, +72 h', 'City PM2.5 -> AQI at the target time', '3 outputs', 'standardised for training'],
], columns=['Branch', 'Feature', 'Definition', 'Shape per forecast', 'Transform'])
display(feat)
table_figure(feat, 'fig10_feature_table', 'Table 2. DualAQI inputs and target', col_w=[1.1, 1.9, 3.6, 1.7, 1.9], row_h=0.36)

In [ ]:
hourly_all = D.load_inputs('data')
fire_s = D.load_fire('data/punjab_fire_daily.csv'); frp_s = D.load_fire('data/punjab_fire_daily.csv', 'frp_sum')
S = D.build_samples(hourly_all, [3], fire=fire_s, event_opts=('frp',), frp=frp_s)
Xb, ev, Y, O = S['Xb'], S['event'], S['Y'], S['origins']
last = D.last_usable_fire_day(O)
tr_daily = D.daily_transport(hourly_all)
F = pd.DataFrame({
    'AQI now': Xb['aqi_now'].values, 'PM2.5 24 h mean': Xb['pm_mean24'].values, 'PM2.5 7-day mean': Xb['pm_mean168'].values,
    'Wind speed': Xb['ws'].values, 'Temperature': Xb['t2m'].values, 'RH': Xb['rh'].values, 'Pressure': Xb['sp'].values,
    'BLH': Xb['blh'].values, 'BLH 24 h mean': Xb['blh_mean24'].values,
    'NW wind (day-1)': tr_daily.reindex(last).values,
    'Fires (day-1)': ev[:, 0], 'Fires (7-day)': np.log1p(np.expm1(ev[:, :7]).sum(1)), 'FRP (day-1)': ev[:, 7],
    'Month (cos)': np.cos(2 * np.pi * (O.month - 1) / 12),
    'AQI +24 h': Y[24].values, 'AQI +48 h': Y[48].values, 'AQI +72 h': Y[72].values})
C_ = F.corr(method='spearman')
fig, ax = plt.subplots(figsize=(9.2, 8.0))
ax.grid(False)
im = ax.imshow(C_.values, cmap=DIVERGE, norm=TwoSlopeNorm(0, -1, 1))
ax.set_xticks(range(len(C_))); ax.set_xticklabels(C_.columns, rotation=55, ha='right')
ax.set_yticks(range(len(C_))); ax.set_yticklabels(C_.columns)
for i in range(len(C_)):
    for j in range(len(C_)):
        v = C_.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6.6, color='white' if abs(v) > 0.6 else INK)
for k in (13.5,):
    ax.axhline(k, color=INK, lw=1); ax.axvline(k, color=INK, lw=1)
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02); cb.set_label('Spearman correlation', color=INK2); cb.outline.set_visible(False)
ax.set_title(f'Feature correlations at the forecast origin (03:00 UTC), {len(O):,} daily origins 2019-2025\n'
             'Last three rows/columns: targets', fontsize=10.5)
save(fig, 'fig11_feature_correlation')

## 2. Model

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.6))
ax.set_xlim(0, 125); ax.set_ylim(0, 60); ax.axis('off')
def box(x, y, w, h, text, fc, ec, fs=8.2, bold_first=True):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.4,rounding_size=1.2', fc=fc, ec=ec, lw=1.1, zorder=3))
    lines = text.split('\n')
    ax.text(x + w / 2, y + h / 2 + (len(lines) - 1) * 1.05, lines[0], ha='center', va='center', fontsize=fs + 0.6,
            fontweight='bold' if bold_first else 'normal', zorder=4)
    for i, t in enumerate(lines[1:], 1):
        ax.text(x + w / 2, y + h / 2 + (len(lines) - 1) * 1.05 - i * 2.4, t, ha='center', va='center', fontsize=fs,
                color=INK2, zorder=4)
def arrow(x0, y0, x1, y1, label=None, lx=0, ly=1.4, color=INK2):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='-|>', color=color, lw=1.2,
                mutation_scale=11, shrinkA=0, shrinkB=0), zorder=2)
    if label:
        ax.text((x0 + x1) / 2 + lx, (y0 + y1) / 2 + ly, label, ha='center', fontsize=8, color=INK, style='italic', zorder=4)
# branch backgrounds
ax.add_patch(FancyBboxPatch((0.5, 33), 49, 24, boxstyle='round,pad=0.3,rounding_size=2', fc='#eef5fd', ec='none', zorder=0))
ax.add_patch(FancyBboxPatch((0.5, 3), 49, 24, boxstyle='round,pad=0.3,rounding_size=2', fc='#fdf1ea', ec='none', zorder=0))
ax.add_patch(FancyBboxPatch((53, 3), 37, 54, boxstyle='round,pad=0.3,rounding_size=2', fc='#f4f3ef', ec='none', zorder=0))
ax.text(1.5, 55, 'SEQUENCE BRANCH', fontsize=8.5, fontweight='bold', color=BLUE)
ax.text(1.5, 25, 'EVENT BRANCH', fontsize=8.5, fontweight='bold', color=ORANGE)
ax.text(54, 55, 'GATED FUSION', fontsize=8.5, fontweight='bold', color=INK2)
box(2, 36.5, 22, 15, 'Hourly input\n168 h x 8 features\nAQI, wind speed, t2m, RH,\nu10, v10, pressure, BLH', 'white', BLUE)
box(30, 39.5, 16, 9, 'LSTM\n8 -> 64 hidden\n18,944 params', '#cde2fb', BLUE)
box(2, 6.5, 22, 15, 'Daily input (v1: 17, v2: 24)\n7 x log(1 + fire count)\n7 x log(1 + FRP)   [v2]\n10 calendar terms', 'white', ORANGE)
box(30, 9.5, 16, 9, 'MLP\nin -> 64 -> 32, ReLU\n3,232 / 3,680 params', '#fbdccc', ORANGE)
box(57, 27, 14, 10, 'Gate\ng = sigmoid(W [h; e])\n96 -> 64, 6,208 params', 'white', INK2, fs=7.8)
box(57, 9.5, 14, 8, 'Projection P\n32 -> 64\n2,112 params', 'white', INK2, fs=7.8)
ax.add_patch(plt.Circle((82, 32), 4.6, fc='white', ec=INK, lw=1.2, zorder=3))
ax.text(82, 32.8, 'fused', ha='center', va='center', fontsize=8.5, fontweight='bold', zorder=4)
ax.text(82, 30.6, 'g*h + (1-g)*Pe', ha='center', va='center', fontsize=6.6, color=INK2, zorder=4)
box(94, 26, 14, 12, 'Output head\nLinear 64->64, ReLU\nDropout 0.3\nLinear 64->3', 'white', INK, fs=7.8)
for i, (lab, y) in enumerate([('AQI t+24 h', 42), ('AQI t+48 h', 32), ('AQI t+72 h', 22)]):
    box(112, y - 3, 11, 6, lab, '#f0efec', INK, fs=7.8)
    arrow(108.6, 32, 111.6, y)
arrow(24.5, 44, 29.6, 44)
arrow(24.5, 14, 29.6, 14)
arrow(46.5, 44, 77.6, 34.6, 'h (64)', lx=6, ly=3)
arrow(52, 41.6, 56.6, 34.6)
arrow(46.5, 14, 56.6, 13.5, 'e (32)', lx=0, ly=1.6)
arrow(52, 14.1, 56.6, 30)
arrow(71.6, 32, 77.3, 32, 'g', ly=1.0)
arrow(71.6, 13.5, 79.2, 28.4, 'Pe', lx=1.6, ly=0)
arrow(86.7, 32, 93.6, 32)
ax.text(62.5, 0.6, 'Total: 34,851 parameters (v1), 35,299 (v2); single-branch LSTM without the event branch: 23,299. '
        'Loss: MSE weighted 0.25 / 0.35 / 0.40 by horizon; Adam 1e-3, 15 epochs, batch 128.',
        ha='center', fontsize=7.8, color=INK2)
ax.set_title('DualAQI architecture', fontsize=12, loc='left')
save(fig, 'fig12_architecture')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.9))
ax.set_xlim(-185, 90); ax.set_ylim(-0.6, 4.2); ax.grid(False)
for s in ('left', 'top', 'right'):
    ax.spines[s].set_visible(False)
ax.set_yticks([])
ax.barh(3.4, 168, left=-168, height=0.5, color='#cde2fb', edgecolor=BLUE)
ax.text(-84, 3.4, 'PM2.5 / AQI + ERA5 sequence: 168 hourly steps up to the origin', ha='center', va='center', fontsize=8.5)
for k in range(1, 8):
    x0 = -3 - 24 * k
    ax.barh(2.3, 24, left=x0, height=0.5, color='#fbdccc', edgecolor=ORANGE, lw=0.9)
    ax.text(x0 + 12, 2.3, f'D-{k}', ha='center', va='center', fontsize=8)
ax.text(-180, 2.3, 'fire days', ha='right', va='center', fontsize=8.5, color=ORANGE)
ax.text(-180, 3.4, 'sequence', ha='right', va='center', fontsize=8.5, color=BLUE)
ax.barh(2.3, 24, left=-3, height=0.5, color='white', edgecolor=MUTED, hatch='////', lw=0.8)
ax.text(9, 2.3, 'day D', ha='center', va='center', fontsize=8, color=MUTED)
ax.text(24, 1.65, 'fire day D is complete at 24:00 UTC,\nso it is used only from D+1 03:00 UTC', fontsize=7.8, color=INK2, ha='left')
ax.axvline(0, color=INK, lw=1.4)
ax.text(1.5, 4.05, 'forecast origin t: 03:00 UTC = 08:30 IST', fontsize=8.5, fontweight='bold')
for h in (24, 48, 72):
    ax.plot(h, 0.9, marker='D', ms=8, color=INK)
    ax.text(h, 0.35, f't+{h} h', ha='center', fontsize=8.5)
ax.text(48, 1.35, 'targets (AQI)', ha='center', fontsize=8.5)
ax.set_xticks(range(-168, 73, 24)); ax.set_xlabel('hours relative to the forecast origin')
ax.set_title('What each forecast may use: causal input windows', fontsize=11.5, loc='left')
save(fig, 'fig13_forecast_timeline')

In [ ]:
folds = D.forward_folds(O, O[0] + pd.Timedelta(days=365))
fig, ax = plt.subplots(figsize=(11.5, 6.2))
ax.grid(True, axis='x')
period_col = lambda k: BLUE if k < 8 else (ORANGE if k < 16 else AQUA)
for k, (tr, te) in enumerate(folds):
    t0, t1, s0, s1 = O[tr[0]], O[tr[-1]], O[te[0]], O[te[-1]]
    ax.barh(k, (t1 - t0).days, left=t0, height=0.62, color='#d9d7d0', lw=0)
    ax.barh(k, (s1 - s0).days + 1, left=s0, height=0.62, color=period_col(k), lw=0)
ax.axvspan(pd.Timestamp('2024-01-01'), pd.Timestamp('2024-07-01'), color='#f0efec', zorder=0, lw=0)
ax.text(pd.Timestamp('2024-04-01'), -1.6, 'no BLH\nin source', ha='center', fontsize=7.8, color=INK2)
ax.set_yticks(range(len(folds))); ax.set_yticklabels([f'fold {k}' for k in range(len(folds))], fontsize=7.6)
ax.invert_yaxis(); ax.set_ylim(len(folds) - 0.4, -2.4)
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#d9d7d0', label='training (expanding window)'),
                   Patch(color=BLUE, label='development test, 2020-2021 (v2 chosen here)'),
                   Patch(color=ORANGE, label='clean test 1, 2022-2023'),
                   Patch(color=AQUA, label='clean test 2, Jul 2024-Dec 2025')],
          loc='lower left', fontsize=8)
ax.set_title('Forward cross-validation: 90-day test blocks; training ends 168 h + 72 h before each block', loc='left', fontsize=11)
save(fig, 'fig14_cv_scheme')

## 3. Results

In [ ]:
def read_oof(path, model, folds=None, rename=None):
    o = pd.read_csv(path, parse_dates=['origin'])
    o = o[(o.scheme == 'forward') & (o.model == model)]
    if folds is not None:
        o = o[o.fold.isin(list(folds))]
    return o.assign(valid=o.origin + pd.to_timedelta(o.horizon, unit='h'), model=rename or model)
def v2_oof(s):
    return pd.concat([read_oof(f'results/dualaqi_v2/dev/C2/seed{s}/oof_predictions.csv.gz', 'C2', range(8), 'DualAQI v2'),
                      read_oof(f'results/dualaqi_v2/test/V2/seed{s}/oof_predictions.csv.gz', 'DualAQI-v2', range(8, 16), 'DualAQI v2')])
v1_oof = lambda s: read_oof(f'results/dualaqi/fire/seed{s}/oof_predictions.csv.gz', 'DualAQI', rename='DualAQI v1')
lstm_oof = lambda s: read_oof(f'results/dualaqi/nofire/seed{s}/oof_predictions.csv.gz', 'DualAQI-noFire', rename='LSTM (single-branch)')
base = pd.read_csv('results/oof_predictions.csv.gz', parse_dates=['origin'])
base = base[base.scheme == 'forward'].assign(valid=lambda d: d.origin + pd.to_timedelta(d.horizon, unit='h'))
V2 = {s: v2_oof(s) for s in (0, 1, 2)}; V1 = {s: v1_oof(s) for s in (0, 1, 2)}; L = {s: lstm_oof(s) for s in (0, 1, 2)}
bench = pd.read_csv('results/benchmark_long_with_dualaqi.csv'); bench = bench[bench.scheme == 'forward']
print('forward-CV forecasts per horizon:', V2[0][V2[0].horizon == 24].shape[0])

In [ ]:
dev = pd.read_csv('results/dualaqi_v2/dev_selection.csv')
label = {'v1': 'DualAQI v1', 'C2': 'v1 + FRP  (selected = v2)', 'D2': 'v1 + wind transport', 'E2': 'v1 + 14/30-day fire sums',
         'F2': 'v1 + FRP + transport + sums', 'B': 'v1 + early stopping', 'C': 'B + FRP', 'D': 'B + transport',
         'E': 'B + fire sums', 'F': 'B + all three', 'LSTM-ES': 'LSTM + early stopping', 'LSTM-v1': 'LSTM (single-branch)'}
def col(m):
    return BLUE if m == 'C2' else AQUA if m == 'v1' else ORANGE if m.startswith('LSTM') else (MUTED if m in 'BCDEF' else INK)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), sharey=True, gridspec_kw=dict(wspace=0.08))
order = dev.sort_values('RMSE_mean_over_horizons', ascending=False).model.tolist()
for ax, h in zip(axes, (24, 48, 72)):
    ax.grid(True, axis='x'); ax.grid(False, axis='y')
    for i, m in enumerate(order):
        r = dev[dev.model == m].iloc[0]
        ax.errorbar(r[f'RMSE_{h}'], i, xerr=r[f'RMSE_{h}_sd'], fmt='o', ms=6, color=col(m), ecolor=col(m), elinewidth=1.2, capsize=0)
        ax.text(r[f'RMSE_{h}'] + r[f'RMSE_{h}_sd'] + 0.4, i, f"{r[f'RMSE_{h}']:.1f}", va='center', fontsize=7.6, color=INK2)
    ax.set_title(f'{h} h ahead', fontsize=10)
    ax.set_xlabel('RMSE (AQI), mean ± SD of 3 seeds')
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels([label[m] for m in order])
suptitle(fig, 'Choosing DualAQI v2 on the development years (forward folds 0-7, 2020-2021)',
         'Grey: early-stopping variants (dropped: worse for both models). Blue: selected v2. Rules fixed in PLAN.md before the runs.', y=1.06)
save(fig, 'fig15_v2_selection')

In [ ]:
models = ['Persistence', 'ARIMA', 'Ridge', 'GBM', 'XGBoost', 'RandomForest', 'Ensemble', 'LSTM', 'DualAQI v1', 'DualAQI v2']
cmap_m = lambda m: MC.get(m, OTHER)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4), gridspec_kw=dict(wspace=0.45))
for ax, h in zip(axes, (24, 48, 72)):
    t = bench[bench.horizon == h].set_index('model').loc[models].sort_values('RMSE', ascending=False)
    ax.grid(True, axis='x'); ax.grid(False, axis='y')
    ax.hlines(range(len(t)), t.RMSE.min() - 3, t.RMSE, color=GRID, lw=1)
    ax.scatter(t.RMSE, range(len(t)), s=46, color=[cmap_m(m) for m in t.index], zorder=3, edgecolor='white', lw=0.8)
    for i, (m, v) in enumerate(t.RMSE.items()):
        ax.text(v + 0.6, i, f'{v:.1f}', va='center', fontsize=7.8, color=INK2)
    ax.set_yticks(range(len(t))); ax.set_yticklabels(t.index)
    for lbl in ax.get_yticklabels():
        if lbl.get_text().startswith('DualAQI'):
            lbl.set_fontweight('bold')
    ax.set_xlim(t.RMSE.min() - 3, t.RMSE.max() + 6)
    ax.set_title(f'{h} h ahead', fontsize=10); ax.set_xlabel('RMSE (AQI)')
suptitle(fig, 'Benchmark: forward cross-validation 2020-2023 (1,440 forecasts per horizon, seed 0)',
         'Lower is better. DualAQI v2 includes the 2020-2021 years used to select it (slightly optimistic).', y=1.07)
save(fig, 'fig16_benchmark_rmse')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4), sharey=True, gridspec_kw=dict(wspace=0.08))
mods = [m for m in models if m != 'Persistence']
for ax, h in zip(axes, (24, 48, 72)):
    t = bench[bench.horizon == h].set_index('model').loc[mods]
    ax.grid(True, axis='x'); ax.grid(False, axis='y')
    ax.axvline(0, color=INK2, lw=0.9)
    for i, m in enumerate(mods):
        r = t.loc[m]; d = r.RMSE - bench[(bench.horizon == h) & (bench.model == 'Persistence')].RMSE.iloc[0]
        ax.hlines(i, r.dRMSE_vs_persist_CI95_lo, r.dRMSE_vs_persist_CI95_hi, color=cmap_m(m), lw=2.4)
        ax.plot(d, i, 'o', color=cmap_m(m), ms=6, mec='white', mew=0.8)
    ax.set_title(f'{h} h ahead', fontsize=10); ax.set_xlabel('RMSE difference vs Persistence (AQI)')
axes[0].set_yticks(range(len(mods))); axes[0].set_yticklabels(mods)
suptitle(fig, 'Skill against Persistence with 95% bootstrap intervals (forward CV 2020-2023, seed 0)',
         'Left of zero = better than Persistence. Intervals resample whole days.', y=1.07)
save(fig, 'fig17_skill_vs_persistence')

In [ ]:
periods = [('results/dualaqi_v2/test_metrics.csv', 'Clean test 1: 2022-2023 (720 forecasts)'),
           ('results/dualaqi_v2/test2024_metrics.csv', 'Clean test 2: Jul 2024-Dec 2025 (523 forecasts)')]
show = [('DualAQI v2', BLUE, -0.18), ('DualAQI v1', AQUA, 0.0), ('LSTM', ORANGE, 0.18)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw=dict(wspace=0.22))
for ax, (path, title) in zip(axes, periods):
    t = pd.read_csv(path); t = t[t.season == 'all']
    for m, c, dx in show:
        r = t[t.model == m].sort_values('horizon')
        x = np.arange(3) + dx
        ax.plot(x, r.RMSE_mean, color=c, lw=1.0, alpha=0.6)
        ax.errorbar(x, r.RMSE_mean, yerr=r.RMSE_sd, fmt='o', color=c, ms=6, capsize=3, elinewidth=1.3,
                    label=m if m != 'LSTM' else 'LSTM (single-branch)')
    ax.set_xticks(range(3)); ax.set_xticklabels(['24 h', '48 h', '72 h'])
    ax.set_ylabel('RMSE (AQI), mean ± SD of 3 seeds'); ax.set_title(title, fontsize=10)
axes[0].legend(loc='upper left')
suptitle(fig, 'Held-out tests: DualAQI v2 vs DualAQI v1 vs single-branch LSTM',
         'Both periods were run once, after all design choices were fixed.', y=1.06)
save(fig, 'fig18_test_periods_rmse')

In [ ]:
comp_col = {'DualAQI v2 - LSTM': BLUE, 'DualAQI v2 - DualAQI v1': AQUA, 'DualAQI v1 - LSTM': ORANGE}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0), sharey=True, gridspec_kw=dict(wspace=0.06))
for ax, (path, title) in zip(axes, [('results/dualaqi_v2/test_drmse.csv', 'Clean test 1: 2022-2023'),
                                    ('results/dualaqi_v2/test2024_drmse.csv', 'Clean test 2: Jul 2024-Dec 2025')]):
    t = pd.read_csv(path); t = t[t.season == 'all']
    ax.grid(True, axis='x'); ax.grid(False, axis='y'); ax.axvline(0, color=INK, lw=0.9)
    ylabels, y = [], 0
    for comp, c in comp_col.items():
        for h in (24, 48, 72):
            r = t[(t.comparison == comp) & (t.horizon == h)].iloc[0]
            ax.hlines(y, r.CI95_7dblock_lo, r.CI95_7dblock_hi, color=c, lw=1.0, alpha=0.7)
            ax.hlines(y, r.CI95_day_lo, r.CI95_day_hi, color=c, lw=3.2)
            sig = r.excludes_zero_day and r.excludes_zero_7dblock
            ax.plot(r.dRMSE, y, 'o', ms=7, color=c if sig else 'white', mec=c, mew=1.6, zorder=3)
            ylabels.append(f'{comp}, {h} h'); y += 1
        y += 0.6; ylabels.append('')
    ax.set_title(title, fontsize=10); ax.set_xlabel('paired RMSE difference (AQI); negative = first model better')
ticks = [i - 0.6 * (i // 4) for i in range(len(ylabels))]
pos, lab = [], []
yy = 0
for comp in comp_col:
    for h in (24, 48, 72):
        pos.append(yy); lab.append(f'{comp}   {h} h'); yy += 1
    yy += 0.6
axes[0].set_yticks(pos); axes[0].set_yticklabels(lab); axes[0].invert_yaxis()
suptitle(fig, 'Paired comparisons with 95% intervals (3 seeds averaged)',
         'Thick: day bootstrap; thin: 7-day block bootstrap. Filled point = both intervals exclude zero.', y=1.06)
save(fig, 'fig19_paired_differences')

In [ ]:
o = V2[0]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True, gridspec_kw=dict(wspace=0.08))
for ax, h in zip(axes, (24, 48, 72)):
    d = o[o.horizon == h]
    ax.grid(False)
    hb = ax.hexbin(d.y_true, d.y_pred, gridsize=38, extent=(0, 600, 0, 600), mincnt=1, cmap=BLUES, linewidths=0)
    ax.plot([0, 600], [0, 600], color=INK, lw=0.9, ls='--')
    for e in CAT_EDGES[1:-1]:
        ax.axvline(e, color=GRID, lw=0.6, zorder=0); ax.axhline(e, color=GRID, lw=0.6, zorder=0)
    r = np.sqrt(np.mean((d.y_pred - d.y_true) ** 2)); r2 = 1 - np.sum((d.y_pred - d.y_true) ** 2) / np.sum((d.y_true - d.y_true.mean()) ** 2)
    ax.text(15, 570, f'RMSE {r:.1f}\nR² {r2:.3f}\nn = {len(d):,}', va='top', fontsize=8.5)
    ax.set_xlim(0, 600); ax.set_ylim(0, 600); ax.set_aspect('equal')
    ax.set_title(f'{h} h ahead', fontsize=10); ax.set_xlabel('observed AQI')
axes[0].set_ylabel('DualAQI v2 forecast AQI')
cb = fig.colorbar(hb, ax=axes, shrink=0.8, pad=0.015); cb.set_label('forecasts per cell', color=INK2); cb.outline.set_visible(False)
suptitle(fig, 'Observed vs forecast AQI, DualAQI v2 (forward CV 2020-2023, seed 0)',
         'Dashed: perfect forecast. Grid lines: AQI category boundaries.', y=1.04)
save(fig, 'fig20_observed_vs_forecast')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6.4), sharex=True, gridspec_kw=dict(hspace=0.25))
for ax, h in zip(axes, (24, 72)):
    obs = V2[0][(V2[0].horizon == h)].set_index('valid').y_true.loc['2023']
    ax.plot(obs.index, obs.values, color=INK, lw=1.4, label='observed', zorder=4)
    for name, src, c in [('DualAQI v2', V2[0], BLUE), ('LSTM (single-branch)', L[0], ORANGE)]:
        s_ = src[src.horizon == h].set_index('valid').y_pred.loc['2023']
        ax.plot(s_.index, s_.values, color=c, lw=1.1, label=name, alpha=0.95)
    ax.set_ylim(0, 600); aqi_bands(ax, 600)
    ax.set_ylabel('AQI'); ax.set_title(f'{h} h ahead', fontsize=10)
axes[0].legend(loc='upper center', ncol=3)
axes[1].xaxis.set_major_locator(mdates.MonthLocator()); axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
suptitle(fig, 'Forecasts through 2023 (clean test 1), one forecast per day at 08:30 IST',
         'Seed 0. The figure shows the valid (target) date.', y=1.0)
save(fig, 'fig21_timeseries_2023')

In [ ]:
def monthly_rmse(o, h):
    d = o[o.horizon == h]
    return d.groupby(d.valid.dt.month).apply(lambda g: np.sqrt(np.mean((g.y_pred - g.y_true) ** 2)))
series = [('DualAQI v2', V2[0], BLUE, '-'), ('LSTM (single-branch)', L[0], ORANGE, '-'),
          ('Ensemble', base[base.model == 'Ensemble'], YELLOW, '-'), ('Persistence', base[base.model == 'Persistence'], MUTED, '--')]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True, gridspec_kw=dict(wspace=0.06))
for ax, h in zip(axes, (24, 72)):
    ax.axvspan(9.5, 11.5, color='#f0efec', zorder=0, lw=0)
    for name, o_, c, ls in series:
        m = monthly_rmse(o_, h)
        ax.plot(m.index, m.values, color=c, ls=ls, marker='o', ms=3.5, label=name)
        ax.text(12.25, m.iloc[-1], name.split(' (')[0], fontsize=7.6, color=INK2, va='center')
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    ax.set_xlim(0.6, 13.6); ax.set_title(f'{h} h ahead', fontsize=10)
axes[0].set_ylabel('RMSE (AQI)'); axes[0].legend(loc='upper center', ncol=2, fontsize=8)
suptitle(fig, 'Forecast error by month (forward CV 2020-2023, seed 0); grey band: Oct-Nov burning season', y=1.03)
save(fig, 'fig22_monthly_rmse')

In [ ]:
def bias_by_cat(runs, h):
    out = []
    for s, o in runs.items():
        d = o[o.horizon == h]
        c = np.digitize(d.y_true, CAT_EDGES[1:-1])
        out.append(pd.Series(d.y_pred.values - d.y_true.values).groupby(c).mean())
    a = pd.concat(out, axis=1)
    return a.mean(1).reindex(range(6)), a.std(1).reindex(range(6))
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True, gridspec_kw=dict(wspace=0.06))
for ax, h in zip(axes, (24, 48, 72)):
    ax.axhline(0, color=INK, lw=0.9)
    for (name, runs, c), dx in zip([('DualAQI v2', V2, BLUE), ('DualAQI v1', V1, AQUA), ('LSTM (single-branch)', L, ORANGE)], (-0.2, 0, 0.2)):
        m, sd = bias_by_cat(runs, h)
        ax.errorbar(np.arange(6) + dx, m, yerr=sd, fmt='o-', color=c, ms=5, lw=1.1, capsize=2, label=name)
    ax.set_xticks(range(6)); ax.set_xticklabels(CATS, rotation=30, ha='right')
    ax.set_title(f'{h} h ahead', fontsize=10)
axes[0].set_ylabel('bias: forecast - observed (AQI)'); axes[0].legend(loc='lower left', fontsize=8)
n = np.bincount(np.digitize(V2[0][V2[0].horizon == 24].y_true, CAT_EDGES[1:-1]), minlength=6)
suptitle(fig, 'Bias by observed AQI category (forward CV 2020-2023, mean ± SD of 3 seeds)',
         'Forecasts per category: ' + ', '.join(f'{c} {k}' for c, k in zip(CATS, n)), y=1.06)
save(fig, 'fig23_bias_by_category')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw=dict(wspace=0.35))
for ax, h in zip(axes, (24, 72)):
    d = V2[0][V2[0].horizon == h]
    cy, cp = np.digitize(d.y_true, CAT_EDGES[1:-1]), np.digitize(d.y_pred, CAT_EDGES[1:-1])
    cm = pd.crosstab(cy, cp).reindex(index=range(6), columns=range(6), fill_value=0)
    pct = cm.div(cm.sum(1).replace(0, np.nan), axis=0) * 100
    ax.grid(False)
    im = ax.imshow(pct.values, cmap=BLUES, vmin=0, vmax=100)
    for i in range(6):
        for j in range(6):
            v = pct.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color='white' if v > 55 else INK)
    ax.set_xticks(range(6)); ax.set_xticklabels(CATS, rotation=35, ha='right')
    ax.set_yticks(range(6)); ax.set_yticklabels(CATS)
    ax.set_xlabel('forecast category'); ax.set_ylabel('observed category')
    acc = np.mean(cy == cp); within = np.mean(np.abs(cy - cp) <= 1)
    ax.set_title(f'{h} h ahead: exact {acc*100:.0f}%, within one category {within*100:.0f}%', fontsize=9.5)
    for s in ax.spines.values():
        s.set_visible(False)
cb = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02); cb.set_label('% of observed category (row)', color=INK2); cb.outline.set_visible(False)
suptitle(fig, 'AQI category confusion, DualAQI v2 (forward CV 2020-2023, seed 0)', y=1.02)
save(fig, 'fig24_category_confusion')

### 3.1 How the fusion gate uses the fire branch
The gate value *g* is the weight on the sequence (LSTM) state; **1 - g** is the share taken from the event (fire + calendar)
branch. This cell retrains DualAQI v2 (seed 0) on every forward fold to read the gate for each test forecast
(about 5 minutes; cached in `results/figures/objective1/gate_values_v2_seed0.csv`). Predictions are identical to the
stored v2 runs because training is deterministic.

In [ ]:
import torch
GATE = f'{OUT}/gate_values_v2_seed0.csv'
if not os.path.exists(GATE):
    rows = []
    for k, (tr, te) in enumerate(folds):
        seq_tr, seq_te = D.standardize(S['seq'][tr], S['seq'][te])
        ev_tr, ev_te = D.standardize_rows(S['event'][tr], S['event'][te])
        holder = []
        def make():
            net = D.DualAQI(S['seq'].shape[2], S['event'].shape[1], use_fire=True)
            holder.append(net)
            return net
        p = D.train_nn(make, [seq_tr, ev_tr], S['Y'].values[tr].astype(np.float32), [seq_te, ev_te], 0, reseed_after_init=True)
        net = holder[0]; net.eval()
        with torch.no_grad():
            _, (hn, _) = net.lstm(torch.from_numpy(seq_te)); hh = hn[-1]
            e = net.event(torch.from_numpy(ev_te))
            gg = torch.sigmoid(net.gate(torch.cat([hh, e], dim=1))).mean(1).numpy()
        rows.append(pd.DataFrame({'origin': O[te], 'fold': k, 'gate_seq': gg, 'pred24': np.clip(p[:, 0], 0, None)}))
        print(f'fold {k}: done', flush=True)
    pd.concat(rows).to_csv(GATE, index=False)
gv = pd.read_csv(GATE, parse_dates=['origin'])
chk = V2[0][V2[0].horizon == 24].set_index('origin').y_pred
common = gv.set_index('origin').pred24.reindex(chk.index).dropna()
print(f'check: retrained predictions match stored v2 runs, max |diff| = {np.abs(common - chk.loc[common.index]).max():.4f}')

In [ ]:
gv['event_share'] = 1 - gv.gate_seq
gv['fires'] = fire_s.reindex(D.last_usable_fire_day(gv.origin)).values
fig, (a, b) = plt.subplots(1, 2, figsize=(11.5, 4.3), gridspec_kw=dict(wspace=0.25))
m = gv.origin.dt.month
med, q1, q3 = gv.event_share.groupby(m).median(), gv.event_share.groupby(m).quantile(0.25), gv.event_share.groupby(m).quantile(0.75)
a.axvspan(9.5, 11.5, color='#f0efec', zorder=0, lw=0)
a.fill_between(med.index, q1, q3, color=ORANGE, alpha=0.2, lw=0)
a.plot(med.index, med.values, color=ORANGE, marker='o', ms=4)
a.set_xticks(range(1, 13)); a.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
a.set_ylabel('event-branch share in the fusion (1 - g)'); a.set_title('a  By month (median, IQR)')
on_ = gv.origin.dt.month.isin([10, 11])
b.grid(True, axis='both')
b.scatter(np.log1p(gv.fires[~on_]), gv.event_share[~on_], s=9, color=MUTED, alpha=0.45, lw=0, label='rest of year')
b.scatter(np.log1p(gv.fires[on_]), gv.event_share[on_], s=12, color=ORANGE, alpha=0.8, lw=0, label='Oct-Nov')
ticks = [0, 10, 100, 1000, 5000]
b.set_xticks(np.log1p(ticks)); b.set_xticklabels([f'{t:,}' for t in ticks])
b.set_xlabel('Punjab fire detections on the last available day'); b.set_ylabel('event-branch share (1 - g)')
b.legend(loc='upper left'); b.set_title('b  Against fire activity')
suptitle(fig, 'How much the gated fusion draws on the fire/calendar branch (DualAQI v2, seed 0, all forward test forecasts 2020-2025)', y=1.03)
save(fig, 'fig25_gate_analysis')

### 3.2 Results summary table

In [ ]:
def row(path, m):
    t = pd.read_csv(path); t = t[(t.season == 'all') & (t.model == m)].sort_values('horizon')
    return ' / '.join(f'{a:.1f}' for a in t.RMSE_mean)
sd = pd.read_csv('results/dualaqi_seed_summary_forward.csv')
b = lambda m: ' / '.join(f"{bench[(bench.model == m) & (bench.horizon == h)].RMSE.iloc[0]:.1f}" for h in (24, 48, 72))
res = pd.DataFrame([
    ['DualAQI v2', b('DualAQI v2'), row('results/dualaqi_v2/test_metrics.csv', 'DualAQI v2'), row('results/dualaqi_v2/test2024_metrics.csv', 'DualAQI v2')],
    ['DualAQI v1', b('DualAQI v1'), row('results/dualaqi_v2/test_metrics.csv', 'DualAQI v1'), row('results/dualaqi_v2/test2024_metrics.csv', 'DualAQI v1')],
    ['LSTM (single-branch)', '-', row('results/dualaqi_v2/test_metrics.csv', 'LSTM'), row('results/dualaqi_v2/test2024_metrics.csv', 'LSTM')],
    ['LSTM (baseline)', b('LSTM'), '-', '-'],
    ['Ensemble', b('Ensemble'), '-', '-'],
    ['Persistence', b('Persistence'), '-', '-'],
], columns=['Model', 'Forward CV 2020-2023\n(seed 0)', 'Clean test 2022-2023\n(3-seed mean)', 'Clean test Jul 2024-2025\n(3-seed mean)'])
display(res)
table_figure(res, 'fig26_results_summary', 'Table 3. RMSE (AQI) at 24 / 48 / 72 h', col_w=[2.0, 2.2, 2.2, 2.4], row_h=0.45)
print('\nall figures:'); print('\n'.join(sorted(os.path.basename(p) for p in glob.glob(f'{OUT}/*.png'))))